# 01 — Data Prep (D1–D2)

**VinDr-CXR → YOLO format.** Positive-only subset, WBF rater fusion, stratified 80/10/10.

> ⚠️ **This notebook is where the paper quietly dies.** If rater fusion is wrong, every downstream number is wrong, and the D4 sanity gate will *not* catch it — broken-but-plausible labels give a broken-but-plausible 0.25 mAP that looks fine.

**Settings:** accelerator `None` (CPU only), internet ON.

**Gate:** §9 `verify()` must pass before notebook 02.

---
### Fusion threshold — settled 2026-07-20

Three runs. IoU **0.5** under-merged badly (2655 duplicate pairs). **0.4** improved it but still left 1331 pairs across 942 images — 21% of the dataset, visually confirmed as 8×-nested *Pulmonary fibrosis* on one image. Final value **0.25**, chosen by measured sweep:

| thr | boxes | retained | dup_pairs |
|---|---|---|---|
| **0.25** ← chosen | **21473** | **59.5%** | **58** |
| 0.30 | 21840 | 60.5% | 413 |
| 0.40 | 22724 | 63.0% | 1331 |
| 0.50 | 23948 | 66.3% | 2655 |
| *floor, all trios merged* | *~12032* | *33.3%* | — |

Duplicates halve per step while retention moves only 6.8 points across the entire range — the threshold trades duplicate labels for almost nothing.

**Over-merging risk checked and cleared.** The classes exposed to spurious merging of genuinely adjacent lesions barely moved from 0.4 → 0.25: Nodule/Mass 70.9→69.2%, Calcification 75.5→72.7%, Cardiomegaly 43.0→42.6%, Aortic enlargement 44.8→43.4%. None approached the 33.3% floor.

**Do not sweep lower.** 58 residual pairs is 0.27% of boxes — noise, likely genuinely adjacent lesions.

In [ ]:
!pip install -q ensemble-boxes

REPO = "/kaggle/working/repo"
!rm -rf {REPO} && git clone -q https://github.com/ShMazumder/vinbigdata-chest-x-ray.git {REPO}

import sys
sys.path.insert(0, REPO)

from pathlib import Path
import pandas as pd, numpy as np

from src import config as C
from src.data import fusion, prepare

!cd {REPO} && git rev-parse --short HEAD    # commit hash -> paper
print(C.as_dict())

## 1. Load raw annotations

Paths live in `config.py`. If they need changing, change them **there** and re-push — not here. The moment values get patched in the notebook, the frozen-protocol guarantee is gone.

> `train.csv` carries two coordinate systems. `x_min/y_min/x_max/y_max` are already scaled to the 1024px JPGs — use those. The `raw_*` columns are original DICOM space; using them against the resized images would silently give boxes 2–3× too large.

In [ ]:
IMAGE_DIR = C.IMAGE_DIR
assert C.TRAIN_CSV.exists(), f"not found: {C.TRAIN_CSV} -- fix config.py"

df_raw = prepare.load_raw(C.TRAIN_CSV)
df_raw.head()

## 2. Rater distribution

**Measured:** R9/R10/R8 produced **34,462 of 36,096 positive boxes = 95.5%**. Only 11 of 17 raters appear at all; R1 and R3–R7 contribute nothing, R2 contributes 3 boxes.

Stronger than `dataset-research-notes.md` originally claimed. **Belongs in the paper's limitations** — the fused labels are effectively three radiologists' opinions, not a 17-rater consensus. It also weakens any "3 independent raters" framing, including P5's.

In [ ]:
if "rad_id" in df_raw.columns:
    vc = df_raw[df_raw.class_id != 14].rad_id.value_counts()
    print(vc)
    top3 = vc.head(3).sum()
    print(f"\ntop-3 raters: {top3}/{vc.sum()} = {top3/vc.sum():.1%} of positive boxes")

## 3. Positive-only subset

Expect **4,394 images (29.3%)**. The compute enabler — ~3.4× epoch-time reduction, and the established protocol here (`sunghyunjun` trained its detector on abnormal images only).

**Report this.** The model never sees normal images and cannot be evaluated on normal-vs-abnormal discrimination.

In [ ]:
df_pos = prepare.positive_only(df_raw)
dims = prepare.get_image_dims(IMAGE_DIR, df_pos.image_id.unique())
print(f"{len(dims)} images measured")

## 4. Fusion threshold sweep

Kept in the pipeline as a reproducibility artifact — the paper can state the threshold was selected by measurement, which most VinDr work cannot.

**Reading it:** `dup_pairs` should fall sharply as the threshold drops; `n_boxes` must not collapse toward the printed floor. Expect this to reproduce the table in the header.

In [ ]:
sweep = fusion.sweep_fusion_iou(df_pos, dims, thresholds=(0.25, 0.3, 0.4, 0.5))
sweep

## 5. Fuse at the frozen threshold

**Unsettled, state it in the paper:** the `sunghyunjun` reference's own ablation has WBF *winning* on private LB (0.185 vs 0.181) while *losing* on CV (0.4158 vs 0.4419). The author chose NMS by reasoning about test-set labeling, not measurement. Don't inherit either as established.

In [ ]:
df_fused = fusion.fuse_dataset(df_pos, dims, method=C.FUSION_METHOD, iou_thr=C.FUSION_IOU)

audit = fusion.count_near_duplicates(df_fused, iou_thr=C.DUPLICATE_AUDIT_IOU)
print(f"\npost-fusion audit: {audit['dup_pairs']} near-duplicate pairs "
      f"across {audit['dup_images']} images")
print("worst classes:", {C.CLASSES[k]: v for k, v in list(audit['dup_per_class'].items())[:5]})

## 6. Fusion report

**Expected at 0.25** (from the settled run — deviation means something changed upstream):

| Class | retained |
|---|---|
| Cardiomegaly / Aortic enlargement | 42.6% / 43.4% |
| Pneumothorax | 51.8% |
| Pleural effusion | 57.5% |
| Pulmonary fibrosis | 61.0% |
| Nodule/Mass | 69.2% |
| Calcification | 72.7% |
| Atelectasis | 78.1% |
| **overall** | **59.5%** |

Anatomically anchored findings merge hardest — raters agree where the heart is. Diffuse findings merge least — raters disagree on extent.

In [ ]:
rep = fusion.fusion_report(df_pos, df_fused)
rep

## 7. Visual check (MANDATORY)

Targeted at the **worst offenders**, not random samples. Random sampling is why runs 1 and 2 nearly passed — the random panels looked clean while the worst images had boxes stacked 8 deep.

Red dotted = rater boxes. Green = fused.

**Looking for:** duplicate near-identical green boxes of the same class in one region.

**Not a problem:** same-class boxes in *different* anatomical regions (ILD in both lungs). Genuinely distinct findings, IoU ≈ 0, no threshold should merge them.

At 0.25 expect ~6 images flagged out of 4,394. Check whether they are real duplicates or genuinely adjacent lesions — if the latter, the pipeline is done.

In [ ]:
worst = fusion.find_duplicate_images(df_fused, iou_thr=C.DUPLICATE_AUDIT_IOU, limit=6)
print(f"{len(worst)} images still showing duplicates" if worst else "no duplicates found ✓")
if worst:
    fig = fusion.plot_fusion_examples(df_pos, df_fused, IMAGE_DIR, worst)

In [ ]:
# Random sample -- confirms fusion didn't over-merge elsewhere.
rng = np.random.default_rng(C.SEED)
sample_ids = list(rng.choice(df_fused.image_id.unique(), size=6, replace=False))
fig = fusion.plot_fusion_examples(df_pos, df_fused, IMAGE_DIR, sample_ids)

## 8. Stratified split + YOLO export

Stratified on each image's **rarest** present class — protects tail classes from vanishing out of val/test, which would make their per-class AP undefined and break the Axis-A analysis.

> ⚠️ **Decide the reporting rule before you see any scores.** Pneumothorax lands at ~10 test boxes and Atelectasis ~22. AP on ten boxes is meaningless — one detection moves it ~10 points — and since mAP averages over all classes with GT, that noise propagates into the headline number.
>
> **Rule for this paper:** keep all 14 classes in mAP (excluding them would break comparability with the 0.253 / 0.338 / 0.415 published figures), report per-class **with `n` visible**, and state the n<30 caveat in limitations. Choosing after seeing scores looks like cherry-picking.

In [ ]:
splits = prepare.stratified_split(df_fused, split=C.SPLIT, seed=C.SEED)
dist = prepare.class_distribution(df_fused, splits)
dist

In [ ]:
thin = dist[dist.test < 30][["class", "train", "val", "test"]]
if len(thin):
    print("⚠ classes with <30 test boxes -- per-class AP unstable, report with n:")
    print(thin)

In [ ]:
# copy_images=True by default -- symlinks do not survive Save & Run All, and
# verify() would still pass on dangling links. ~0.93 GB, a few minutes.
C.DATA_ROOT.mkdir(parents=True, exist_ok=True)
prepare.to_yolo_labels(df_fused, dims, splits, C.DATA_ROOT, IMAGE_DIR)
data_yaml = prepare.write_data_yaml(C.DATA_ROOT, C.CLASSES)

## 9. Verify — GATE

Must print `✓ verify passed` **and** a non-zero dataset size (~0.93 GB). `0.00 GB` means symlinks — the saved dataset would be broken.

In [ ]:
ok = prepare.verify(C.DATA_ROOT, splits)
assert ok, "data prep failed verification -- do NOT proceed to training"

df_fused.to_csv(C.DATA_ROOT / "fused_boxes.csv", index=False)
pd.Series(splits).to_json(C.DATA_ROOT / "splits.json")

# Provenance -- these numbers go straight into the paper's dataset section.
import json
(C.DATA_ROOT / "prep_manifest.json").write_text(json.dumps({
    **C.as_dict(),
    "n_raw_rows": len(df_raw),
    "n_positive_images": len(dims),
    "n_raw_boxes": int((df_pos.class_id != 14).sum()),
    "n_fused_boxes": len(df_fused),
    "duplicate_audit": audit,
    "split_sizes": {k: len(v) for k, v in splits.items()},
    "fusion_sweep": sweep.to_dict(orient="records"),
}, indent=2, default=str))

print(f"\ndone. data.yaml -> {data_yaml}")

## 10. Commit for notebook 02

**Save Version → "Save & Run All (Commit)"** — *not* Quick Save. Only a full commit persists `/kaggle/working` as notebook output.

Push to GitHub **first**: the commit re-clones the repo, so anything edited interactively but not pushed won't be in the committed run.

Then in notebook 02: **Add Data** → **"Notebook Output Files"** tab → filter **"Your Work"** → select this notebook. `C.find_data_yaml()` locates it from there.

Keep the notebook **private** — VinDr-CXR carries a data use agreement restricting redistribution.